In [ ]:
# !pip install --upgrade transformers datasets evaluate

In [1]:
import re
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    matthews_corrcoef,
    balanced_accuracy_score,
)
from transformers import (
    AutoConfig,
    AutoModel,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    default_data_collator,
)
from datasets import (
    load_dataset, 
    ClassLabel, 
)
import evaluate

/usr/local/lib/python3.10/dist-packages/torchvision/datapoints/__init__.py:14: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
/usr/local/lib/python3.10/dist-packages/torchvision/transforms/v2/__init__.py:64: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https:/

In [18]:
MODEL_NAME = "distilbert-base-uncased"

TEST_SIZE = 0.2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 2
BATCH_SIZE = 16

DATA_PATH = "data/sentence_sets_trimmed.csv"
DATA_ENCODING = "ISO-8859-1"

LABEL_COLUMN = "applicant_gender"
TEXT_COLUMN = "s1_s2"

STRATIFY_ENABLED = False
DEGENDER_ENABLED = True
BALANCED_WEIGHTS_ENABLED = False

MAX_LENGTH = 512

OUTPUT_NAME = f"{MODEL_NAME}-finetuned-nlp-letters-{TEXT_COLUMN}"
OUTPUT_NAME += "-degendered" if DEGENDER_ENABLED else ""
OUTPUT_NAME += "-stratified" if STRATIFY_ENABLED else ""
OUTPUT_NAME += "-balanced" if BALANCED_WEIGHTS_ENABLED else ""

In [19]:
class LettersBERTModule(nn.Module):
    def __init__(self, model_name, num_labels, class_weights=None, cls_or_mean="cls"):
        super().__init__()

        self.config = AutoConfig.from_pretrained(model_name)
        self.config.num_labels = num_labels
        self.config.class_weights = class_weights
        self.config.cls_or_mean = cls_or_mean

        # Layers
        self.transformer = AutoModel.from_pretrained(model_name, config=self.config)
        self.classifier = nn.Linear(self.config.hidden_size, self.config.num_labels)

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)

        # Either use [CLS] token or mean pooling
        if self.config.cls_or_mean == "mean":
            output = torch.mean(outputs.last_hidden_state, dim=1)
        else:
            output = outputs.last_hidden_state[:, 0, :]

        # Logits and loss
        logits = self.classifier(output)
        loss = None

        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.config.class_weights)
            loss = loss_fct(logits, labels)

        return {"loss": loss, "logits": logits}

In [20]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [21]:
def degender(text, level="original"):
    if level == "enhanced":
        titles = r"\b(?:mr|mrs|ms|miss|mister|sir|madam)\b"
        nouns = r"\b(?:man|men|woman|women|gentleman|lady|boy|boys|girl|girls)(?:'s)?\b"
        pronouns = r"\b(?:he|him|his|she|her|hers|himself|herself)\b"
    else:
        titles = r"\b(?:mr|mrs|ms|miss|mister)\b"
        nouns = r"\b(?:man|men|woman|women|gentleman|lady)(?:'s)?\b"
        pronouns = r"\b(?:he|him|his|she|her|hers)\b"

    titles_regex = re.compile(titles, flags=re.IGNORECASE)
    nouns_regex = re.compile(nouns, flags=re.IGNORECASE)
    pronouns_regex = re.compile(pronouns, flags=re.IGNORECASE)

    text = titles_regex.sub("mx", text)
    text = nouns_regex.sub("person", text)
    text = pronouns_regex.sub("they", text)

    return text

In [22]:
def preprocess(data):
    texts = data[TEXT_COLUMN]

    if DEGENDER_ENABLED:
        texts = [degender(t) for t in texts]

    tokenized = tokenizer(
        texts, 
        truncation=True, 
        padding=True
    )

    tokenized["labels"] = data[LABEL_COLUMN]

    return tokenized

In [23]:
# Load the data
dataset = load_dataset("csv", data_files=DATA_PATH, encoding=DATA_ENCODING)

# Recast labels as class features
unique_labels = dataset["train"].unique(LABEL_COLUMN)

features = dataset["train"].features
features[LABEL_COLUMN] = ClassLabel(names=unique_labels)

dataset = dataset.cast(features)

dataset = dataset["train"].train_test_split(
    test_size=TEST_SIZE,
    stratify_by_column=LABEL_COLUMN if STRATIFY_ENABLED else None,
    seed=100,
)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [24]:
print([degender(t) for t in dataset["train"][TEXT_COLUMN][:10]])

['I have been asked to provide a letter of recommendation for FIRST_NAME LAST_NAME in support of they application to your residency program  * FIRST_NAME is a fourth   year medical student at the lubbock campus of the texas tech school of medicine  * they spent a month in our department as LAST_NAME elective rotation  * I had many chances to visit with they and work with they   so I feel that I LAST_NAME adequately assess they character and work ethic  * FIRST_NAME approached me early in they fourth   year rotation to express they interest in our specialty  * we had time to have several meaningful discussions and I feel that FIRST_NAME is choosing our specialty for all of the right reasons  * academically   FIRST_NAME is a very solid candidate  * they has also scored well MIDDLE_NAME the usmle with a 237 MIDDLE_NAME step I anda 236 MIDDLE_NAME step il  * in addition   they has worked successfully MIDDLE_NAME several academic projects with a great deal of success  * MIDDLE_NAME a person

In [25]:
train_dataset = train_dataset.map(preprocess, batched=True)
test_dataset = test_dataset.map(preprocess, batched=True)

Map:   0%|          | 0/2628 [00:00<?, ? examples/s]

Map:   0%|          | 0/657 [00:00<?, ? examples/s]

In [26]:
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")
confusion_metric = evaluate.load("confusion_matrix")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    accuracy = accuracy_metric.compute(predictions=preds, references=labels)
    precision = precision_metric.compute(predictions=preds, references=labels, average="macro")
    recall = recall_metric.compute(predictions=preds, references=labels, average="macro")
    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")

    mcc = matthews_corrcoef(labels, preds)
    bal_acc = balanced_accuracy_score(labels, preds)
    
    cr = classification_report(labels, preds, target_names=train_dataset.features[LABEL_COLUMN].names)
    cm = confusion_metric.compute(predictions=preds, references=labels)

    print("Confusion Matrix:\n", cm["confusion_matrix"])
    print("Classification Report:\n", cr)
    print("MCC:", mcc)
    print("Balanced Accuracy:", bal_acc)

    return {
        "accuracy": accuracy["accuracy"],
        "precision": precision["precision"],
        "recall": recall["recall"],
        "f1": f1["f1"],
        "mcc": mcc,
        "balanced_accuracy": bal_acc,
    }

In [27]:
# Class weight balancing
class_weights = torch.tensor(
    compute_class_weight(
        "balanced", 
        classes=np.unique(train_dataset[LABEL_COLUMN]), 
        y=train_dataset[LABEL_COLUMN]
    ), dtype=torch.float
)

In [28]:
# model = LettersBERTModule(model_name=MODEL_NAME, num_labels=len(LABELS), class_weights=class_weights)

In [29]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=len(train_dataset.features[LABEL_COLUMN].names)
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [30]:
# Training arguments
training_args = TrainingArguments(
    output_dir=f"./{OUTPUT_NAME}",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
)

/home/hice1/mwesley32/.local/lib/python3.10/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/tmp/ipykernel_515708/1892133964.py:17: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [31]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Mcc,Balanced Accuracy
1,0.606600,0.585297,0.727549,0.363775,0.500000,0.421145,0.000000,0.500000
2,0.538400,0.489235,0.794521,0.869971,0.626400,0.640546,0.432501,0.626400


Confusion Matrix:
 [[478   0]
 [179   0]]
Classification Report:
               precision    recall  f1-score   support

        male       0.73      1.00      0.84       478
      female       0.00      0.00      0.00       179

    accuracy                           0.73       657
   macro avg       0.36      0.50      0.42       657
weighted avg       0.53      0.73      0.61       657

MCC: 0.0
Balanced Accuracy: 0.5


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1497: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/m

Confusion Matrix:
 [[476   2]
 [133  46]]
Classification Report:
               precision    recall  f1-score   support

        male       0.78      1.00      0.88       478
      female       0.96      0.26      0.41       179

    accuracy                           0.79       657
   macro avg       0.87      0.63      0.64       657
weighted avg       0.83      0.79      0.75       657

MCC: 0.4325006760336266
Balanced Accuracy: 0.6263995699025269


TrainOutput(global_step=330, training_loss=0.5836716883110278, metrics={'train_runtime': 100.7977, 'train_samples_per_second': 52.144, 'train_steps_per_second': 3.274, 'total_flos': 696248647335936.0, 'train_loss': 0.5836716883110278, 'epoch': 2.0})

In [32]:
trainer.evaluate()

Confusion Matrix:
 [[476   2]
 [133  46]]
Classification Report:
               precision    recall  f1-score   support

        male       0.78      1.00      0.88       478
      female       0.96      0.26      0.41       179

    accuracy                           0.79       657
   macro avg       0.87      0.63      0.64       657
weighted avg       0.83      0.79      0.75       657

MCC: 0.4325006760336266
Balanced Accuracy: 0.6263995699025269


{'eval_loss': 0.48923519253730774,
 'eval_accuracy': 0.7945205479452054,
 'eval_precision': 0.8699712643678161,
 'eval_recall': 0.6263995699025269,
 'eval_f1': 0.6405456557068113,
 'eval_mcc': 0.4325006760336266,
 'eval_balanced_accuracy': 0.6263995699025269,
 'eval_runtime': 3.8931,
 'eval_samples_per_second': 168.758,
 'eval_steps_per_second': 10.788,
 'epoch': 2.0}